<a href="https://colab.research.google.com/github/Hanzet22/TKJ-Dumps/blob/main/Projek_Harfi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests

In [ ]:
# ================================================
#  🛡️ Router Vulnerability Scanner v1.0
#  Produk TKJ - Karya: Harfi
#  Fungsi: Cari celah keamanan router sederhana
#  ⚠️ HANYA UNTUK EDUKASI & JARINGAN SENDIRI!
# ================================================

import socket
import requests
import subprocess
import platform
from datetime import datetime
import ipaddress

print("""
   ╔═══════════════════════════════════════════════════╗
   ║   🛡️  Router Vulnerability Scanner v1.0           ║
   ║   Cari Celah Keamanan Router Sederhana            ║
   ╚═══════════════════════════════════════════════════╝
""")

print("⚠️ PERINGATAN:")
print("   Tool ini hanya untuk EDUKASI dan pengujian di JARINGAN SENDIRI.")
print("   Gunakan hanya dengan izin pemilik jaringan!")
print("   Penulis tidak bertanggung jawab atas penyalahgunaan.\n")

# ================================================
# 1. DETEKSI ROUTER DEFAULT
# ================================================
def detect_router():
    """Coba deteksi gateway default"""
    routers = ["192.168.1.1", "192.168.0.1", "192.168.100.1", "10.0.0.1", "192.168.88.1"]

    print("🔍 Mencari router...")
    for ip in routers:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((ip, 80))
            sock.close()
            if result == 0:
                print(f"✅ Router ditemukan: {ip}")
                return ip
        except:
            pass

    # Coba dari gateway sistem
    try:
        if platform.system().lower() == "windows":
            output = subprocess.run(["ipconfig"], capture_output=True, text=True)
            for line in output.stdout.split('\n'):
                if "Default Gateway" in line:
                    ip = line.split(":")[-1].strip()
                    if ip:
                        print(f"✅ Router ditemukan: {ip}")
                        return ip
        else:
            output = subprocess.run(["ip", "route"], capture_output=True, text=True)
            for line in output.stdout.split('\n'):
                if "default via" in line:
                    ip = line.split(" ")[2]
                    if ip:
                        print(f"✅ Router ditemukan: {ip}")
                        return ip
    except:
        pass

    print("❌ Router tidak ditemukan.")
    return None

# ================================================
# 2. SCAN PORT UMUM ROUTER
# ================================================
def scan_router_ports(ip):
    """Scan port yang sering dipakai router"""
    common_ports = {
        21: "FTP",
        22: "SSH",
        23: "Telnet",
        53: "DNS",
        80: "HTTP (Web Admin)",
        443: "HTTPS (Web Admin)",
        8080: "HTTP-Alt (Admin)",
        8443: "HTTPS-Alt (Admin)",
        161: "SNMP",
        1900: "UPnP",
        7547: "TR-069 (Remote Management)",
        5555: "ADB (Android Debug Bridge)",
    }

    print(f"\n🔍 Scan port umum di {ip}...")
    open_ports = []

    for port, service in common_ports.items():
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((ip, port))
            sock.close()
            if result == 0:
                print(f"   ✅ Port {port} ({service}) terbuka")
                open_ports.append((port, service))
            else:
                print(f"   ❌ Port {port} ({service}) tertutup", end='\r')
        except:
            pass

    return open_ports

# ================================================
# 3. CEK WEB ADMIN
# ================================================
def check_web_admin(ip):
    """Coba akses web admin router"""
    print(f"\n🌐 Cek web admin di {ip}...")

    for port, protocol in [(80, "http"), (443, "https"), (8080, "http")]:
        try:
            url = f"{protocol}://{ip}:{port}"
            r = requests.get(url, timeout=3, allow_redirects=True)
            print(f"   ✅ {url} — Status: {r.status_code}")

            # Cek header server
            server = r.headers.get('Server', 'N/A')
            print(f"   📡 Server: {server}")

            # Cek apakah ada login page
            if "login" in r.text.lower() or "admin" in r.text.lower():
                print(f"   ⚠️  Kemungkinan halaman login admin!")
            return True
        except:
            print(f"   ❌ {protocol}://{ip}:{port} — Tidak bisa diakses")

    return False

# ================================================
# 4. CEK DEFAULT CREDENTIAL (HANYA EDUKASI!)
# ================================================
def check_default_creds(ip):
    """Cek apakah router masih pake default credential"""
    # Ini hanya simulasi, tidak melakukan login sebenarnya!
    print("\n🔑 Cek default credential (simulasi):")
    print("   ⚠️  Tidak melakukan login, hanya mengecek celah umum.")

    default_creds = {
        "admin": "admin",
        "admin": "password",
        "admin": "1234",
        "root": "root",
        "user": "user",
        "admin": "12345",
    }

    print(f"\n   💡 Jika router masih pake default credential, hacker bisa masuk.")
    print(f"   💡 Coba cek di: http://{ip} atau https://{ip}")
    print(f"   💡 Username/Password default yang sering dipakai:")
    for user, pw in list(default_creds.items())[:5]:
        print(f"      - {user} / {pw}")

    return True

# ================================================
# 5. CEK TR-069 / REMOTE MANAGEMENT
# ================================================
def check_tr069(ip):
    """Cek apakah TR-069 (port 7547) terbuka"""
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(2)
        result = sock.connect_ex((ip, 7547))
        sock.close()
        if result == 0:
            print("\n⚠️  Port 7547 (TR-069) TERBUKA!")
            print("   TR-069 adalah protokol remote management ISP.")
            print("   Jika dibuka, ISP bisa mengakses router Anda dari jarak jauh.")
            print("   💡 Matikan jika tidak diperlukan.")
            return True
        else:
            print("\n✅ Port 7547 (TR-069) tertutup. Aman.")
            return False
    except:
        print("\n❌ Gagal cek TR-069.")
        return False

# ================================================
# 6. REKOMENDASI
# ================================================
def show_recommendations(open_ports, ip):
    print("\n" + "="*50)
    print("📋 REKOMENDASI KEAMANAN:")
    print("="*50)

    risks = []

    if any(p[0] in [23, 21] for p in open_ports):
        risks.append("🔴 FTP/Telnet terbuka (tidak aman!)")

    if any(p[0] in [80, 443] for p in open_ports):
        risks.append("🟡 Web admin terbuka, ganti password default!")

    if any(p[0] == 7547 for p in open_ports):
        risks.append("🔴 TR-069 terbuka, ISP bisa akses router!")

    if any(p[0] == 22 for p in open_ports):
        risks.append("🟡 SSH terbuka, gunakan key authentication!")

    if not risks:
        print("✅ Router terlihat aman. Tetap rutin update firmware.")
    else:
        for r in risks:
            print(f"   {r}")

    print(f"\n💡 Saran umum:")
    print("   1. Ganti password default router!")
    print("   2. Matikan remote management (TR-069) jika tidak perlu.")
    print("   3. Update firmware router secara rutin.")
    print("   4. Matikan UPnP jika tidak perlu.")
    print("   5. Gunakan firewall.")

    print(f"\n📋 Waktu scan: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("✨ Scan selesai!")

# ================================================
# MAIN PROGRAM
# ================================================
def main():
    # Deteksi router
    ip = detect_router()

    if not ip:
        # Minta input manual
        ip = input("\n🎯 Masukkan IP router manual: ").strip()
        if not ip:
            print("❌ Tidak ada IP. Keluar.")
            return

    # Validasi IP
    try:
        ipaddress.ip_address(ip)
    except:
        print("❌ IP tidak valid!")
        return

    print(f"\n🎯 Target: {ip}")

    # Scan port
    open_ports = scan_router_ports(ip)

    # Cek web admin
    check_web_admin(ip)

    # Cek TR-069
    check_tr069(ip)

    # Cek default credential (edukasi)
    check_default_creds(ip)

    # Rekomendasi
    show_recommendations(open_ports, ip)

if __name__ == "__main__":
    main()